In [1]:
import sys
import os
sys.path.append(os.path.abspath('..')) # Go up one level to the project root and add it to the path
from src.etl.ingest import load_nvdrs
from src.etl.transform import filter_nvdrs_suicides, aggregate_nvdrs_daily
import matplotlib.pyplot as plt
import math
from darts import TimeSeries
from darts.models import ExponentialSmoothing, AutoARIMA, LightGBMModel
import pickle

In [ ]:
# If you just want to see the whole data config dictionary
#from src.utils.config import load_config
# data_config = load_config("data.yaml")


### checks undetermined intent
# nvdrs_df['IncidentCategory_c'] == 'Single death of undetermined intent'
'''
usecols = ['IncidentID', 'IncidentYear', 'SiteID', 'IncidentNumber', 'NarrativeCME','NarrativeLE','IncidentCategory_c', 'HomicideSuicide_c', 'PersonID', 'VictimNumber', 
    'PersonType', 'NumberWeapons_c', 'NumberSuspects_c', 'NumberSubstances_c', 'NumberSubstancesCausedDeath_c', 'Sex', 'AgeYears_c', 'Country', 'ResidenceState', 
    'ResidenceFIPS', 'ResidenceCityState', 'ResidenceZip', 'Homeless', 'AbstractorDeathmanner_c', 'InjuryState', 'InjuryFIPS', 'InjuryCityState', 'InjuryZip', 
    'InjuryDate', 'InjuryDate_myr', 'InjuryDate_year', 'InjuryTime', 'InjuryLocationType', 'RecentRelease', 'AlcoholUseSuspected', 'ExternalCause1ICD9', 
    'ExternalCause2ICD9', 'UnderlyingCauseCode', 'DeathCause1', 'DeathCause2', 'DeathCause3', 'OtherCondition', 'HowInjuryOccurred', 'DeathDate', 'DeathDate_myr', 
    'DeathDate_year', 'DeathState', 'DeathFIPS', 'MultipleConditionCode01ICD10', 'MultipleConditionCode02ICD10', 'MultipleConditionCode03ICD10', 'MultipleConditionCode04ICD10', 
    'MultipleConditionCode05ICD10', 'MultipleConditionCode06ICD10', 'MultipleConditionCode07ICD10', 'MultipleConditionCode08ICD10', 'MultipleConditionCode09ICD10', 
    'MultipleConditionCode10ICD10', 'CME_CircumstancesOtherText', 'CME_CrisisOtherDescription', 'LE_CircumstancesOtherText', 'LE_CrisisOtherDescription', 'SuicideAttemptHistory_c', 
    'SuicideThoughtHistory_c', 'HistorySelfHarm_c']

uselesscols = ['IncidentID','IncidentYear', 'SiteID', 'IncidentNumber','NarrativeCME','NarrativeLE', 'IncidentCategory_c', 'HomicideSuicide_c', 
    'PersonID', 'VictimNumber', 'PersonType', 'NumberSuspects_c', 'Sex', 'AgeYears_c', 'Country', 'AbstractorDeathmanner_c', 'DeathDate', 'DeathDate_myr', 
    'DeathDate_year', 'DeathState', 'DeathFIPS']

und_path = "/Users/matteoperini/Library/CloudStorage/Box-Box/Matteo Perini/6_SuicideProj/CODE/suicide_forecasting/data/raw/und_int.csv"
und_df = pd.read_csv(
    und_path, 
    encoding="cp1252", 
    encoding_errors="replace", # Replaces problematic characters instead of crashing
    low_memory=True, 
    dtype={'DeathDate': str, 'DeathDate_myr': str, 'DeathDate_year': str},
#    nrows=10,
    usecols=uselesscols 
)

#cols_string = ", ".join(uselesscols)
#query = f"SELECT {cols_string} FROM read_csv('{nvdrs_path}', auto_detect=true, ignore_errors=false, encoding='CP1252') WHERE IncidentCategory_c = 'Single death of undetermined intent'"
#und_int = duckdb.query(query).to_df()


#nvdrs_df = load_nvdrs(file_key="nvdrs", data_folder="raw", usecols=uselesscols)
allcols = load_nvdrs(file_key="nvdrs", data_folder="raw", nrows=0)
und_int = nvdrs_df[nvdrs_df['IncidentCategory_c'] == 'Single death of undetermined intent']
nvdrs_df['IncidentCategory_c'].value_counts()
'''

In [ ]:
#usecols = ['IncidentID', 'IncidentYear', 'SiteID', 'IncidentNumber', 'IncidentCategory_c', 'HomicideSuicide_c', 'PersonID', 'VictimNumber', 'PersonType', 'NumberWeapons_c', 'NumberSuspects_c', 'NumberSubstances_c', 'NumberSubstancesCausedDeath_c', 'Sex', 'AgeYears_c', 'Country', 'ResidenceState', 'ResidenceFIPS', 'ResidenceCityState', 'ResidenceZip', 'Homeless', 'AbstractorDeathmanner_c', 'InjuryState', 'InjuryFIPS', 'InjuryCityState', 'InjuryZip', 'InjuryDate', 'InjuryDate_myr', 'InjuryDate_year', 'InjuryTime', 'InjuryLocationType', 'RecentRelease', 'AlcoholUseSuspected', 'ExternalCause1ICD9', 'ExternalCause2ICD9', 'UnderlyingCauseCode', 'DeathCause1', 'DeathCause2', 'DeathCause3', 'OtherCondition', 'HowInjuryOccurred', 'DeathDate', 'DeathDate_myr', 'DeathDate_year', 'DeathState', 'DeathFIPS', 'MultipleConditionCode01ICD10', 'MultipleConditionCode02ICD10', 'MultipleConditionCode03ICD10', 'MultipleConditionCode04ICD10', 'MultipleConditionCode05ICD10', 'MultipleConditionCode06ICD10', 'MultipleConditionCode07ICD10', 'MultipleConditionCode08ICD10', 'MultipleConditionCode09ICD10', 'MultipleConditionCode10ICD10', 'CME_CircumstancesOtherText', 'CME_CrisisOtherDescription', 'LE_CircumstancesOtherText', 'LE_CrisisOtherDescription', 'SuicideAttemptHistory_c', 'SuicideThoughtHistory_c', 'HistorySelfHarm_c']
#uselesscols = ['IncidentID', 'IncidentYear', 'SiteID', 'IncidentNumber', 'IncidentCategory_c', 'HomicideSuicide_c', 'PersonID', 'VictimNumber', 'PersonType', 'NumberSuspects_c', 'Sex', 'AgeYears_c', 'Country', 'AbstractorDeathmanner_c', 'InjuryState', 'InjuryFIPS', 'InjuryCityState', 'InjuryZip', 'InjuryDate', 'InjuryDate_myr', 'InjuryDate_year', 'InjuryTime', 'InjuryLocationType']
usemincol = ['IncidentID','DeathDate', 'DeathDate_myr', 'DeathDate_year', 'DeathState', 'DeathFIPS', 'IncidentCategory_c','PersonType'] 

uselesscols = ['IncidentID','IncidentYear', 'SiteID', 'IncidentNumber','NarrativeCME','NarrativeLE', 'IncidentCategory_c', 'HomicideSuicide_c', 
    'PersonID', 'VictimNumber', 'PersonType', 'NumberSuspects_c', 'Sex', 'AgeYears_c', 'Country', 'AbstractorDeathmanner_c', 'DeathDate', 'DeathDate_myr', 'DeathDate_year', 'DeathState', 'DeathFIPS']

if 'nvdrs_s_df' not in locals():
    nvdrs_df = load_nvdrs(file_key="nvdrs", data_folder="raw", usecols=usemincol )
    # Filter suicides
    nvdrs_s_df = filter_nvdrs_suicides(nvdrs_df)
    # starting from 01/01/2010
    nvdrs_s_df = nvdrs_s_df[nvdrs_s_df['DeathDate'] >= '2010-01-01']
    del nvdrs_df


# Create Univariate TimeSeries (US Total)
nvdrs_s_daily_df = aggregate_nvdrs_daily(nvdrs_s_df)

# from pands df to darts ts (US Total)
nvdrs_ts = TimeSeries.from_dataframe(
    nvdrs_s_daily_df, 
    time_col='DeathDate', 
    value_cols='incident_count',
    fill_missing_dates=True, 
    freq='D',
    fillna_value=0
)


In [ ]:
## STATE
# Create Multivariate TimeSeries (By State)
daily_df = aggregate_nvdrs_daily(nvdrs_s_df, geo_level='state').set_index('DeathDate')


# create monthly (Month End) and Weekly DF
daily_df = daily_df.resample('D').sum() # This adds any missing days and fills with 0
monthly_df = daily_df.resample('ME').sum()
weekly_df = daily_df.resample('W').sum()


# 2. Convert df to Darts TimeSeries
ts_daily = TimeSeries.from_dataframe(daily_df.reset_index(), time_col='DeathDate')
ts_monthly = TimeSeries.from_dataframe(monthly_df.reset_index(), time_col='DeathDate')
ts_weekly = TimeSeries.from_dataframe(weekly_df.reset_index(), time_col='DeathDate')

In [ ]:
# Create Multivariate TimeSeries (By county)
nvdrs_s_df=nvdrs_s_df[nvdrs_s_df['DeathDate'] >= '2020-01-01']



daily_cnty_df = aggregate_nvdrs_daily(nvdrs_s_df, geo_level='county').set_index('DeathDate')


# create monthly (Month End) and Weekly DF
daily_cnty_df = daily_cnty_df.resample('D').sum() # This adds any missing days and fills with 0
monthly_cnty_df = daily_cnty_df.resample('ME').sum()
weekly_cnty_df = daily_cnty_df.resample('W').sum()


# 2. Convert df to Darts TimeSeries
ts_cnty_daily = TimeSeries.from_dataframe(daily_cnty_df.reset_index(), time_col='DeathDate')
ts_cnty_monthly = TimeSeries.from_dataframe(monthly_cnty_df.reset_index(), time_col='DeathDate')
ts_cnty_weekly = TimeSeries.from_dataframe(weekly_cnty_df.reset_index(), time_col='DeathDate')

In [ ]:
selected_states = ['Arizona', 'Iowa', 'Kentucky', 'New Jersey', 'New York', 'North Carolina', 'Utah']

# Filter the monthly dataframe to just these columns and plot
monthly_df[selected_states].plot(figsize=(15, 7))

plt.title("Monthly Incidents for Selected States")
plt.ylabel("Incident Count")
plt.xlabel("Date")
plt.show()



weekly_df[selected_states].plot(figsize=(15, 7))

plt.title("weekly Incidents for Selected States")
plt.ylabel("Incident Count")
plt.xlabel("Date")
plt.show()



daily_df[selected_states].plot(figsize=(15, 7))

plt.title("Daily Incidents for Selected States")
plt.ylabel("Incident Count")
plt.xlabel("Date")
plt.show()

In [ ]:
selected_cntys = {
#    'Arizona': ('Maricopa, AZ', 'Pima, AZ', 'Pinal, AZ', 'Yavapai, AZ', 'Coconino, AZ'),
    'Arizona': ('Pima, AZ', 'Pinal, AZ', 'Yavapai, AZ', 'Coconino, AZ'),  
    'Iowa': ('Polk, IA', 'Linn, IA', 'Scott, IA', 'Johnson, IA', 'Black Hawk, IA'), 
    'Kentucky': ('Jefferson, KY', 'Fayette, KY', 'Kenton, KY', 'Boone, KY', 'Warren, KY'), 
    'New Jersey': ('Bergen, NJ', 'Middlesex, NJ', 'Essex, NJ', 'Hudson, NJ', 'Monmouth, NJ'), 
    'New York': ('New York, NY', 'Kings, NY', 'Queens, NY', 'Bronx, NY', 'Richmond, NY'), 
    'North Carolina': ('Wake, NC', 'Mecklenburg, NC', 'Guilford, NC', 'Forsyth, NC', 'Cumberland, NC'), 
    'Utah': ('Salt Lake, UT', 'Utah, UT', 'Davis, UT', 'Weber, UT', 'Washington, UT')
}

colors = plt.get_cmap('tab10', len(selected_cntys))
state_colors = {state: colors(i) for i, state in enumerate(selected_cntys.keys())}

def plot_counties(df, title):
    plt.figure(figsize=(15, 7))
    
    for state, counties in selected_cntys.items():
        color = state_colors[state]
        for i, county in enumerate(counties):
            # Only label the first county of each state for a clean legend
            label = state if i == 0 else "_nolegend_"
            if county in df.columns:
                plt.plot(df.index, df[county], color=color, label=label)
                
    plt.title(title)
    plt.ylabel("Incident Count")
    plt.xlabel("Date")
    plt.legend(title="State")
    plt.show()

# Generate the three plots
plot_counties(monthly_cnty_df, "Monthly Incidents for Selected Counties")
plot_counties(weekly_cnty_df, "Weekly Incidents for Selected Counties")
plot_counties(daily_cnty_df, "Daily Incidents for Selected Counties")

In [ ]:
# {Name: (TimeSeries, Validation_Length, LGBM_Lags)} # lags=12 means it uses the past 12 months to predict the next month
timeframes = {
    'Monthly': (ts_monthly, 12, 24), # using `LGBM_Lags` months to predict `Validation_Length` month
    'Weekly': (ts_weekly, 12, 24),
    'Daily': (ts_daily, 30, 60)
}

cols = 2
rows = math.ceil(len(selected_states) / cols)

for name, (ts_data, val_len, lags) in timeframes.items():
    plot_len = val_len * 3 
    
    fig, axes = plt.subplots(nrows=rows, ncols=cols, figsize=(15, 4 * rows), constrained_layout=True)
    axes = axes.flatten()
    fig.suptitle(f"{name} Forecasts", fontsize=16)
    
    # --- LightGBM (Multivariate handling) ---
    model_path = f"../artifacts/lgbm_model_{name}_NVDRS.pkl"
    pred_path = f"../artifacts/lgbm_pred_{name}_NVDRS.pkl"
    
    train_multi = ts_data[:-val_len]
    
    # fit and forecast only if the pkl files don't exist
    if os.path.exists(pred_path):
        with open(pred_path, "rb") as f:
            lgbm_pred = pickle.load(f)
    else:
        if os.path.exists(model_path):
            lgbm_model = LightGBMModel.load(model_path)
        else:
            lgbm_model = LightGBMModel(lags=lags)
            lgbm_model.fit(train_multi)
            lgbm_model.save(model_path)
        
        lgbm_pred = lgbm_model.predict(val_len)
        with open(pred_path, "wb") as f:
            pickle.dump(lgbm_pred, f)
            
    # --- Univariate Models & Plotting ---
    for i, state in enumerate(selected_states):
        ts = ts_data[state]
        train = ts[:-val_len]
        
        es_pred = ExponentialSmoothing().fit(train).predict(val_len)
        aa_pred = AutoARIMA().fit(train).predict(val_len)
        lgbm_uni_pred = LightGBMModel(lags=lags).fit(train).predict(val_len)
        
        ts[-plot_len:].plot(ax=axes[i], label='Actual')
        es_pred.plot(ax=axes[i], label='ExponentialSmoothing', lw=2)
        aa_pred.plot(ax=axes[i], label='AutoARIMA', lw=2)
        lgbm_pred[state].plot(ax=axes[i], label='LightGBM (Multi)', lw=2)
        lgbm_uni_pred.plot(ax=axes[i], label='LightGBM (Uni)', lw=2)
        
        axes[i].set_title(state)
        axes[i].legend()
        
    for j in range(len(selected_states), len(axes)):
        axes[j].axis('off')
        
    plt.show()